# Mesh overlay

Overlay operations combine two meshes by cutting one with the other's edges.
Mefikit exposes four families:

| Inputs | Method | Result |
|---|---|---|
| 2D + 2D, first order | `overlay(m2, op)` | `UNION`, `INTERSECTION`, `DIFFERENCE`, `SYMMETRIC_DIFFERENCE` or `IMPRINT` |
| 2D + 2D, quadratic | `overlay(m2, IMPRINT)` | subject refined by the cutter's **circular arcs** |
| 2D + 1D | `intersect_1d(lines)` | subject refined by `SEG2` / `PLINE` edges |
| 2D surfaces in 3D | `overlay_surfaces(m2)` | two mutually conformal surfaces |

*The 2D curved figures are drawn from the exact analytic arcs with Matplotlib;
the boolean, 1D and 3D figures use PyVista. Plotting boilerplate and mesh
builders are hidden. The visible cells contain only the operations: `quad8`
builds a single QUAD8 cell and `mesh_area` returns its exact area including the
circular segments.*

In [ ]:
import numpy as np

import mefikit as mf

OP = mf.OverlayOperation

In [ ]:
import matplotlib.pyplot as plt
import pyvista as pv
from IPython.display import display

pv.set_plot_theme("dark")
pv.set_jupyter_backend("static")


def mesh_area(mesh):
    """Exact area, including the circular segments of curved cells."""
    return sum(float(np.sum(v)) for v in mesh.measure().values())


def quad8(sag, inward=False, shift=(0.0, 0.0)):
    """One QUAD8 cell; `sag` is the bulge of its four edges."""
    dx, dy = shift
    v = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]]) + [dx, dy]
    top = 1.0 - sag if inward else 1.0 + sag
    m = np.array([[0.5, -sag], [1.0 + sag, 0.5], [0.5, top], [-sag, 0.5]]) + [dx, dy]
    mesh = mf.UMesh(np.vstack((v, m)))
    mesh.add_regular_block("QUAD8", np.arange(8, dtype=np.uintp)[None, :])
    return mesh


def quad_surface(extent, n):
    """Flat QUAD4 grid in the plane z = 0."""
    flat = mf.build_cmesh(
        np.linspace(extent[0], extent[1], n + 1),
        np.linspace(extent[2], extent[3], n + 1),
    )
    coords = np.c_[np.asarray(flat.coords()), np.zeros(len(flat.coords()))]
    mesh = mf.UMesh(coords)
    mesh.add_regular_block("QUAD4", np.asarray(flat.blocks()["QUAD4"]))
    return mesh


# --- 2D curved figures: exact analytic arcs, drawn with Matplotlib ---------
# PyVista/VTK mis-renders coplanar translucent surfaces and triangulates
# concave polygons poorly; sampling the exact circles avoids those artefacts.


def arc_outline(p0, m, p1, n=100):
    # Points along the circular arc p0 -> p1 through mid-side node m.
    # A collinear mid node degenerates to a straight segment.
    p0 = np.asarray(p0, float)[:2]
    m = np.asarray(m, float)[:2]
    p1 = np.asarray(p1, float)[:2]
    a, b = m - p0, p1 - p0
    cross = a[0] * b[1] - a[1] * b[0]
    if abs(cross) < 1e-12 * (np.linalg.norm(a) * np.linalg.norm(b) + 1e-30):
        t = np.linspace(0.0, 1.0, n)[:, None]
        return p0 * (1 - t) + p1 * t
    la, lb = a @ a, b @ b
    ctr = p0 + np.array([la * b[1] - lb * a[1], lb * a[0] - la * b[0]]) / (2 * cross)
    r = np.linalg.norm(p0 - ctr)
    a0 = np.arctan2(p0[1] - ctr[1], p0[0] - ctr[0])
    a1 = np.arctan2(p1[1] - ctr[1], p1[0] - ctr[0])
    am = np.arctan2(m[1] - ctr[1], m[0] - ctr[0])
    ccw = (a1 - a0) % (2 * np.pi)
    sweep = ccw if (am - a0) % (2 * np.pi) <= ccw else ccw - 2 * np.pi
    ang = a0 + sweep * np.linspace(0.0, 1.0, n)
    return np.c_[ctr[0] + r * np.cos(ang), ctr[1] + r * np.sin(ang)]


def mesh_outlines(mesh):
    # (outline, vertices) per cell, sampled on the exact arcs.
    # polyze yields QPOLY rings [vertices..., mid-side nodes...] for curved
    # cells and PGON rings [vertices...] for first-order cells.
    poly = mesh.polyze()
    co = np.asarray(poly.coords(), dtype=float)[:, :2]
    cells = []
    for et, conn in poly.blocks().items():
        if not isinstance(conn, tuple):
            continue
        data, offs = np.asarray(conn[0]), np.asarray(conn[1])
        starts = np.r_[0, offs[:-1]]
        for st, o in zip(starts, offs):
            ring = data[st:o]
            if et == "QPOLY":
                nv = len(ring) // 2
                vertices, mids = ring[:nv], ring[nv:]
            else:
                vertices, mids = ring, None
            arcs, vpts = [], []
            for i in range(len(vertices)):
                q0 = co[vertices[i]]
                q1 = co[vertices[(i + 1) % len(vertices)]]
                mid = co[mids[i]] if mids is not None else (q0 + q1) / 2
                arcs.append(arc_outline(q0, mid, q1))
                vpts.append(q0)
            cells.append((np.vstack(arcs), np.asarray(vpts)))
    return cells


def show_curves(*panels, limits=(-0.4, 1.6)):
    # Panels of (title, [(mesh, kwargs), ...]) drawn from exact arcs.
    # kwargs: color (else a distinct colour per cell), alpha, wireframe,
    # linewidth. Overlapping translucent cells are composited in 2D, so the
    # shared arc and the overlap region stay exact.
    fig, axes = plt.subplots(
        1, len(panels), figsize=(6.2 * len(panels), 5.0), squeeze=False
    )
    cmap = plt.get_cmap("tab20")
    for j, (title, layers) in enumerate(panels):
        ax = axes[0][j]
        ax.set_aspect("equal")
        ax.set_facecolor("#1b1b1b")
        for li, (mesh, kw) in enumerate(layers):
            color = kw.get("color")
            alpha = kw.get("alpha", 0.9)
            lw = kw.get("linewidth", 0.9)
            for k, (outline, verts) in enumerate(mesh_outlines(mesh)):
                c = color if color is not None else cmap((2 * k + 1) % 20)
                if kw.get("wireframe"):
                    ax.plot(
                        outline[:, 0],
                        outline[:, 1],
                        color=c,
                        lw=lw,
                        alpha=alpha,
                        zorder=3 + li,
                    )
                else:
                    ax.fill(
                        outline[:, 0],
                        outline[:, 1],
                        color=c,
                        alpha=alpha,
                        edgecolor="white",
                        linewidth=0.8,
                        zorder=2 + li,
                    )
                    ax.plot(
                        verts[:, 0],
                        verts[:, 1],
                        "o",
                        color="white",
                        ms=2.5,
                        alpha=alpha,
                        zorder=5 + li,
                    )
        ax.set_title(title, color="white", fontsize=11, pad=8)
        ax.set_xlim(*limits)
        ax.set_ylim(*limits)
        ax.grid(alpha=0.12)
        ax.tick_params(colors="#888888", labelsize=7)
    fig.patch.set_facecolor("white")
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def coloured(mesh):
    """Colour each cell by its index so that cuts are visible."""
    grid = mesh.to_pyvista()
    grid.cell_data["piece"] = np.arange(grid.n_cells)
    return grid


def show_grid(meshes, labels, shape, cpos="xy"):
    """Grid of (title, mesh) panels."""
    pt = pv.Plotter(shape=shape)
    for i in range(shape[0]):
        for j in range(shape[1]):
            pt.subplot(i, j)
            pt.add_text(labels[i][j], font_size=12)
            m = meshes[i][j]
            pt.add_mesh(
                m if isinstance(m, pv.DataSet) else m.to_pyvista(), show_edges=True
            )
            if cpos:
                pt.camera_position = cpos
    pt.show(cpos=cpos)


def show_panels(*panels, cpos="xy"):
    """Panels of (title, [(grid, kwargs), ...]) layers."""
    pt = pv.Plotter(shape=(1, len(panels)))
    for j, (title, layers) in enumerate(panels):
        pt.subplot(0, j)
        pt.add_text(title, font_size=12)
        for grid, kw in layers:
            pt.add_mesh(grid, **kw)
        if cpos:
            pt.camera_position = cpos
    pt.show(cpos=cpos)

## Boolean overlay (first-order meshes)

Two staggered QUAD4 grids are enough to show the five operations. `IMPRINT`
keeps the subject's domain and refines it with the other mesh's edges; the four
boolean operations select which pieces to keep.

| Operation | Keeps |
|---|---|
| `UNION` | everything covered by at least one mesh |
| `INTERSECTION` | the overlap |
| `DIFFERENCE` | the subject minus the other mesh |
| `SYMMETRIC_DIFFERENCE` | what belongs to exactly one mesh |
| `IMPRINT` | the whole subject, refined |

All five require first-order meshes (`TRI3`, `QUAD4`, `PGON`); passing a
quadratic mesh to a boolean operation raises `ValueError`.

In [ ]:
x = np.linspace(0.0, 3.0, 4)
mesh1 = mf.build_cmesh(x, x)
mesh2 = mf.build_cmesh(x + 1.25, x + 1.25)

results = {
    "Union": mesh1.overlay(mesh2, OP.UNION),
    "Intersection": mesh1.overlay(mesh2, OP.INTERSECTION),
    "Difference": mesh1.overlay(mesh2, OP.DIFFERENCE),
    "Symmetric difference": mesh1.overlay(mesh2, OP.SYMMETRIC_DIFFERENCE),
    "Imprint 2→1": mesh1.overlay(mesh2, OP.IMPRINT),
    "Imprint 1→2": mesh2.overlay(mesh1, OP.IMPRINT),
}

In [ ]:
show_grid(
    [
        [mesh1, mesh2],
        [results["Union"], results["Intersection"]],
        [results["Difference"], results["Symmetric difference"]],
        [results["Imprint 2→1"], results["Imprint 1→2"]],
    ],
    [
        ["Mesh 1", "Mesh 2 (staggered)"],
        ["Union", "Intersection"],
        ["Difference", "Symmetric difference"],
        ["Imprint 2 on 1", "Imprint 1 on 2"],
    ],
    (4, 2),
)

The ragged cell in the middle of *Difference* and *Symmetric difference* is a
PyVista rendering artefact for non-convex polygons, not a mesh defect.

## Quadratic edges are circular arcs

A quadratic 2D cell (`TRI6`, `TRI7`, `QUAD8`, `QUAD9`, `QPOLY`) stores its
boundary as `[vertices…, mid-side nodes…]`: each edge is the circular arc
through its two endpoints and its mid-side node. A collinear mid node
degenerates to a straight segment.

`IMPRINT` accepts quadratic cells on **both** sides. It intersects arcs with
segments or with other arcs and emits touched pieces as `QPOLY` whose mid-side
nodes are recomputed on the original circles — so the exact curved area is
preserved.

In [ ]:
subject = quad8(0.20)
cutter = mf.build_cmesh([-0.25, 0.5, 1.25], [0.25, 0.75])
imprinted = subject.overlay(cutter, OP.IMPRINT)

In [ ]:
before, after = mesh_area(subject), mesh_area(imprinted)
assert np.isclose(before, after, atol=1e-12)
print(f"exact curved area: {before:.9f} → {after:.9f}")
print("output types:", imprinted.block_types())

In [ ]:
show_curves(
    (
        "QUAD8 subject + linear cutter",
        [
            (subject, {"color": "lightblue", "alpha": 0.75}),
            (cutter, {"color": "orange", "wireframe": True, "linewidth": 1.5}),
        ],
    ),
    (
        "QPOLY pieces (exact curved area)",
        [(imprinted, {})],
    ),
)

### Concave arcs

An edge may also bulge *towards the interior* of the cell. That arc sweeps
clockwise instead of counter-clockwise, and the cut must still land on the
right branch of the circle. Concave cells are supported too — including a
cutter edge touching an arc tangentially: the resulting piece is then split at
the pinch point rather than emitted as a self-touching polygon.

In [ ]:
concave = quad8(0.25, inward=True)
concave_cut = concave.overlay(cutter, OP.IMPRINT)
assert np.isclose(mesh_area(concave), mesh_area(concave_cut), atol=1e-12)

In [ ]:
show_curves(
    ("Concave QUAD8", [(concave, {"color": "lightblue", "alpha": 0.9})]),
    ("Refined, area preserved", [(concave_cut, {})]),
)

### Arc ↔ arc: a quadratic cutter

Two different circles meet through their radical axis; two arcs on the same
circle share their overlap endpoints. The cutter's curved edges refine the
subject exactly where the arcs meet, and the new interface follows the
cutter's circular arc — not its chord. Outer arcs and the exact curved area
are preserved.

In [ ]:
other = quad8(0.10, shift=(0.35, 0.30))
arc_cut = subject.overlay(other, OP.IMPRINT)
assert np.isclose(mesh_area(subject), mesh_area(arc_cut), atol=1e-12)
print("refined pieces:", arc_cut.num_elements())

In [ ]:
show_curves(
    (
        "Two curved cells",
        [
            (subject, {"color": "lightblue", "alpha": 0.75}),
            (other, {"color": "orange", "alpha": 0.6}),
        ],
    ),
    ("Subject refined at arc crossings", [(arc_cut, {})]),
)

## Cutting a 2D mesh with a 1D mesh

`intersect_1d` is the 2D/1D counterpart of `IMPRINT`: the subject is refined
with the edges of a 1D mesh of `SEG2` or `PLINE` cells. A `PLINE` is split into
its consecutive `SEG2` edges, so both forms behave identically.

The 1D elements must not intersect one another. A closed polyline lying
strictly inside a cell is ignored — the result would be a cell with a hole,
which is not a valid element.

In [ ]:
plane = mf.build_cmesh([0.0, 1.0], [0.0, 1.0])

segments = mf.UMesh(np.array([[-0.5, 0.5], [0.5, 0.5], [1.5, 0.5]]))
segments.add_regular_block("SEG2", np.array([[0, 1], [1, 2]], dtype=np.uintp))
cut_by_seg2 = plane.intersect_1d(segments)

In [ ]:
zigzag = mf.UMesh(
    np.array([[-0.5, 0.3], [0.2, 0.3], [0.5, 0.7], [0.8, 0.3], [1.5, 0.3]])
)
zigzag.add_poly_block(
    "PLINE", np.arange(5, dtype=np.uintp), np.array([5], dtype=np.uintp)
)
cut_by_pline = plane.intersect_1d(zigzag)
print("SEG2 cutter:", cut_by_seg2.block_types(), cut_by_seg2.num_elements(), "cells")
print("PLINE cutter:", cut_by_pline.block_types(), cut_by_pline.num_elements(), "cells")

In [ ]:
zpts = np.c_[np.asarray(zigzag.coords()), np.zeros(len(zigzag.coords()))]
zline = pv.PolyData(zpts, lines=np.array([5, 0, 1, 2, 3, 4]))

show_panels(
    (
        "SEG2 cutter",
        [
            (
                coloured(cut_by_seg2),
                {
                    "scalars": "piece",
                    "cmap": "tab20",
                    "show_scalar_bar": False,
                    "show_edges": True,
                },
            ),
            (segments.to_pyvista(), {"color": "orange", "line_width": 3}),
        ],
    ),
    (
        "PLINE cutter (zigzag)",
        [
            (
                coloured(cut_by_pline),
                {
                    "scalars": "piece",
                    "cmap": "tab20",
                    "show_scalar_bar": False,
                    "show_edges": True,
                },
            ),
            (zline, {"color": "orange", "line_width": 3}),
        ],
    ),
)

## Surfaces in 3D

`overlay_surfaces` imprints two 2D surfaces embedded in 3D wherever they
coincide. Both must be piecewise planar. The refined surfaces share their
intersection nodes, so they become mutually conformal on the overlap; parts
not covered by the other surface are copied verbatim.

In [ ]:
surface1 = quad_surface((0.0, 2.0, 0.0, 2.0), 4)
surface2 = quad_surface((1.0, 2.8, 0.5, 1.9), 5)
tilt = mf.Transform.rotation([1.0, 0.0, 0.0], -0.6) @ mf.Transform.rotation(
    [0.0, 1.0, 0.0], 0.5
)
surface1, surface2 = surface1.transform(tilt), surface2.transform(tilt)
surfaces = surface1.overlay_surfaces(surface2)

In [ ]:
assert np.array_equal(
    np.asarray(surfaces.refined1.coords()), np.asarray(surfaces.refined2.coords())
)
print("input faces:", len(surfaces.parents1), "+", len(surfaces.parents2))

In [ ]:
pt = pv.Plotter(shape=(1, 2))
pt.subplot(0, 0)
pt.add_text("Two independent surfaces")
pt.add_mesh(surface1.to_pyvista(), show_edges=True)
pt.add_mesh(surface2.to_pyvista(), show_edges=True, color="orange")
pt.subplot(0, 1)
pt.add_text("Refined, mutually conformal")
pt.add_mesh(surfaces.refined1.to_pyvista(), color="lightblue", show_edges=True)
pt.add_mesh(
    surfaces.refined2.to_pyvista(), style="wireframe", color="red", line_width=2
)
pt.show()

## Rules and current limits

- Input meshes must be valid, planar and correctly oriented (CCW), with
  coordinates in the same space.
- **Boolean operations are first-order only.** Their inside/outside
  classification assumes straight-edged cut pieces; passing `TRI6`, `TRI7`,
  `QUAD8`, `QUAD9` or `QPOLY` raises `ValueError`. `IMPRINT` has no such
  restriction and is exact on curved cells.
- A closed cutter loop strictly inside a cell is ignored (no holed cells).
- 1D cutters are assumed not to intersect one another.
- `overlay_surfaces` requires piecewise-planar first-order surfaces.